# Vivida.ge, Google Search opportunity analysis

## tl;dr

- Keep production SEO content and routing unchanged through at least **September 3, 2026**. The last 28 days already show rapid discovery: clicks rose from 1,791 to 3,963 and impressions from 14,414 to 41,952.
- Do not add new pages for the visible opportunity queries yet. Moon calendar, zodiac, IQ, angel number 1111, pregnancy dreams, and zodiac elements already have dedicated indexable pages. Duplicates would risk cannibalization while Google is still reassessing the cluster.
- Recheck the same 28-day comparison after a stable seven-day window. Optimize only pages whose impressions remain substantial while CTR and position stop improving.

## Context & Methods

### Key assumptions

- Source of truth: Google Search Console web-search exports for the `sc-domain:vivida.ge` property, downloaded August 27, 2026.
- The three-month export covers May 26–August 25, 2026. The comparison export covers the latest 28 complete GSC days versus the preceding 28 days.
- Search Console position is impression-weighted and does not prove causality. The export is capped at the rows GSC exposes, while headline totals come from the report itself.
- Opportunity pages were matched to existing routes in the local repository; no production mutation is performed by this notebook.

In [1]:
import csv, io, zipfile
from pathlib import Path

THREE_MONTH_ZIP = Path('/Users/giviendeladze/Desktop/vivida.ge-Performance-on-Search-2026-08-27.zip')
COMPARE_ZIP = Path('/Users/giviendeladze/Desktop/vivida.ge-Performance-on-Search-2026-08-27 (1).zip')

def read_csv_from_zip(zip_path, member):
    with zipfile.ZipFile(zip_path) as archive:
        return list(csv.DictReader(io.StringIO(archive.read(member).decode('utf-8-sig'))))

chart_rows = read_csv_from_zip(THREE_MONTH_ZIP, 'Chart.csv')
weekly = []
for offset in range(0, len(chart_rows), 7):
    chunk = chart_rows[offset:offset + 7]
    if len(chunk) != 7:
        continue
    clicks = sum(int(row['Clicks']) for row in chunk)
    impressions = sum(int(row['Impressions']) for row in chunk)
    weighted_position = sum(float(row['Position']) * int(row['Impressions']) for row in chunk) / impressions
    weekly.append({
        'week_start': chunk[0]['Date'],
        'week_end': chunk[-1]['Date'],
        'clicks': clicks,
        'impressions': impressions,
        'ctr': round(clicks / impressions, 4),
        'position': round(weighted_position, 2),
        'days': len(chunk),
    })

headline = {
    'clicks_current': 3963,
    'clicks_previous': 1791,
    'clicks_change_pct': round((3963 / 1791 - 1) * 100, 1),
    'impressions_current': 41952,
    'impressions_previous': 14414,
    'impressions_change_pct': round((41952 / 14414 - 1) * 100, 1),
    'ctr_current': round(3963 / 41952, 4),
    'ctr_previous': round(1791 / 14414, 4),
    'position_current': 7.2,
    'position_previous': 7.5,
}
print('Headline:', headline)
print('Complete weekly periods:', len(weekly))
print('Latest complete week:', weekly[-1])

Headline: {'clicks_current': 3963, 'clicks_previous': 1791, 'clicks_change_pct': 121.3, 'impressions_current': 41952, 'impressions_previous': 14414, 'impressions_change_pct': 191.1, 'ctr_current': 0.0945, 'ctr_previous': 0.1243, 'position_current': 7.2, 'position_previous': 7.5}
Complete weekly periods: 13
Latest complete week: {'week_start': '2026-08-18', 'week_end': '2026-08-24', 'clicks': 1120, 'impressions': 13720, 'ctr': 0.0816, 'position': 6.75, 'days': 7}


In [2]:
page_rows = read_csv_from_zip(COMPARE_ZIP, 'Pages.csv')
focus_routes = [
    '/mtvaris-kalendari', '/zodiac', '/iq-testi', '/akhali-mtvare',
    '/sizmris-akhsna/orsuloba', '/angeluri-ricxvebi/1111',
    '/zodiac/elementebi', '/zodiac/elementebi/fire',
]
by_route = {row['Top pages'].removeprefix('https://www.vivida.ge'): row for row in page_rows}
for route in focus_routes:
    row = by_route[route]
    print(
        f"{route}: clicks {row['Previous 28 days Clicks']}→{row['Last 28 days Clicks']}; "
        f"impressions {int(row['Previous 28 days Impressions']):,}→{int(row['Last 28 days Impressions']):,}; "
        f"position {float(row['Previous 28 days Position']):.2f}→{float(row['Last 28 days Position']):.2f}"
    )

/mtvaris-kalendari: clicks 16→156; impressions 1,439→6,938; position 8.74→6.63
/zodiac: clicks 4→25; impressions 794→2,762; position 11.51→8.06
/iq-testi: clicks 0→19; impressions 57→587; position 8.77→7.50
/akhali-mtvare: clicks 2→17; impressions 144→581; position 8.72→8.14
/sizmris-akhsna/orsuloba: clicks 0→10; impressions 0→912; position 0.00→8.38
/angeluri-ricxvebi/1111: clicks 0→0; impressions 0→225; position 0.00→7.57
/zodiac/elementebi: clicks 6→6; impressions 431→502; position 7.47→6.52
/zodiac/elementebi/fire: clicks 0→0; impressions 96→183; position 7.33→5.85


## Results

The growth pattern is broad and still accelerating. The latest complete week reached 1,120 clicks and 13,720 impressions, compared with 27 clicks and 526 impressions in the first complete week of the three-month export. Average position improved from 14.54 to 6.75 over the same endpoints.

The largest visible query opportunities already map to dedicated pages. Low CTR is therefore not evidence of missing coverage; it mostly reflects Google expanding those pages into broader result sets. The steepest page-level gains are on `/mtvaris-kalendari`, `/zodiac`, `/iq-testi`, and `/akhali-mtvare`. `/sizmris-akhsna/orsuloba` and `/angeluri-ricxvebi/1111` are newly visible and do not yet have a stable comparison baseline.

A production change now would overlap with recent SEO and international-URL changes from August 22 and make attribution harder.

## Takeaways

1. Freeze production SEO content, metadata, routing, canonicals, and sitemap priority changes for 7–10 days.
2. Re-export the same comparison after September 3. Prioritize a snippet test only if a page keeps at least 300 impressions, remains in positions 4–10, and its CTR is flat or falling across two equal windows.
3. First candidates if the signal persists: `/angeluri-ricxvebi/1111`, `/zodiac/elementebi/fire`, and `/zodiac`; prefer title/description or internal-link tests on the existing canonical URLs over new pages.
4. Keep `/mtvaris-kalendari` unchanged unless its position or clicks reverse: its current gains are too strong to justify a rewrite.